# 🏥 Liver Cancer Classification with MobileNet-V1
## Transfer Learning Approach for 2D CT Slices

**Author:** AI Mentor for Student Project  
**Target:** Kaggle Free Tier GPU (P100/T4)

### Features:
- 📥 Auto-downloads dataset from Kaggle via `kagglehub`
- 🧠 MobileNet-V1 backbone with Transfer Learning
- 🎯 HU Windowing (-75 to 150) + CLAHE preprocessing
- 📊 Data Augmentation (rotation, flipping)
- 📈 Confusion Matrix, DSC, Sensitivity evaluation
- ⚡ Fast training (~15-20 min on Kaggle GPU)

---

In [ ]:
# ============================================================================
# CELL 1: INSTALL DEPENDENCIES (Kaggle-compatible)
# ============================================================================
!pip install -q kagglehub nibabel scikit-image tqdm

print("✅ Dependencies installed!")

: 

In [ ]:
# ============================================================================
# CELL 2: IMPORTS
# ============================================================================
import os
import warnings
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T

import nibabel as nib
from skimage import exposure
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import kagglehub

warnings.filterwarnings('ignore')

# Device setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("\n✅ All imports loaded!")

In [ ]:
# ============================================================================
# CELL 3: CONFIGURATION
# ============================================================================

class Config:
    """Configuration for the liver cancer classifier."""
    
    # Directories
    OUTPUT_DIR = './output'
    MODEL_DIR = './models'
    
    # HU Windowing (as specified: -75 to 150)
    HU_MIN = -75
    HU_MAX = 150
    
    # CLAHE parameters
    CLAHE_CLIP_LIMIT = 0.01
    
    # Model
    IMAGE_SIZE = 224  # MobileNet default input size
    NUM_CLASSES = 3   # Background, Liver, Tumor
    CLASS_NAMES = ['Background', 'Liver', 'Tumor']
    
    # Training (optimized for free GPU tier)
    BATCH_SIZE = 32
    NUM_WORKERS = 2
    EPOCHS = 15
    LEARNING_RATE = 0.001  # As specified
    
    # Data split
    VAL_SPLIT = 0.2
    TEST_SPLIT = 0.1

# Create directories
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
os.makedirs(Config.MODEL_DIR, exist_ok=True)

print("⚙️ Configuration:")
print(f"   • Image Size: {Config.IMAGE_SIZE}x{Config.IMAGE_SIZE}")
print(f"   • HU Window: [{Config.HU_MIN}, {Config.HU_MAX}]")
print(f"   • CLAHE Clip Limit: {Config.CLAHE_CLIP_LIMIT}")
print(f"   • Learning Rate: {Config.LEARNING_RATE}")
print(f"   • Batch Size: {Config.BATCH_SIZE}")
print(f"   • Epochs: {Config.EPOCHS}")

In [ ]:
# ============================================================================
# CELL 4: DOWNLOAD DATASETS FROM KAGGLE
# ============================================================================

print("📥 Downloading Liver Tumor Segmentation datasets...\n")

# Download Part 1
print("[1/2] Downloading: andrewmvd/liver-tumor-segmentation")
try:
    dataset_path_1 = kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation")
    print(f"      ✅ Downloaded to: {dataset_path_1}")
except Exception as e:
    print(f"      ❌ Failed: {e}")
    dataset_path_1 = None

# Download Part 2
print("\n[2/2] Downloading: andrewmvd/liver-tumor-segmentation-part-2")
try:
    dataset_path_2 = kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation-part-2")
    print(f"      ✅ Downloaded to: {dataset_path_2}")
except Exception as e:
    print(f"      ❌ Failed: {e}")
    dataset_path_2 = None

# Collect paths
DATA_PATHS = [p for p in [dataset_path_1, dataset_path_2] if p is not None]
print(f"\n📂 Dataset paths available: {len(DATA_PATHS)}")

In [ ]:
# ============================================================================
# CELL 5: PREPROCESSING FUNCTIONS (HU Windowing + CLAHE)
# ============================================================================

def apply_hu_windowing(image: np.ndarray, hu_min: int = -75, hu_max: int = 150) -> np.ndarray:
    """
    Apply HU (Hounsfield Unit) Windowing to CT image.
    
    This technique enhances liver tissue visibility by clipping
    the HU values to a specific range and normalizing to [0, 1].
    
    Args:
        image: CT image in HU units
        hu_min: Minimum HU value (default: -75 for liver)
        hu_max: Maximum HU value (default: 150 for liver)
    
    Returns:
        Windowed image normalized to [0, 1]
    """
    # Clip to window range
    windowed = np.clip(image, hu_min, hu_max)
    # Normalize to [0, 1]
    windowed = (windowed - hu_min) / (hu_max - hu_min)
    return windowed.astype(np.float32)


def apply_clahe(image: np.ndarray, clip_limit: float = 0.01) -> np.ndarray:
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalization).
    
    CLAHE enhances local contrast while preventing noise amplification,
    making tumor boundaries more visible.
    
    Args:
        image: Input image (normalized to [0, 1])
        clip_limit: Clipping limit for CLAHE (default: 0.01)
    
    Returns:
        CLAHE-enhanced image
    """
    # skimage's equalize_adapthist expects [0, 1] input
    enhanced = exposure.equalize_adapthist(image, clip_limit=clip_limit)
    return enhanced.astype(np.float32)


def preprocess_ct_slice(ct_slice: np.ndarray) -> np.ndarray:
    """
    Full preprocessing pipeline: HU Windowing → CLAHE.
    
    Args:
        ct_slice: Raw CT slice in HU units
    
    Returns:
        Preprocessed image ready for model input
    """
    # Step 1: Apply HU Windowing
    windowed = apply_hu_windowing(ct_slice, Config.HU_MIN, Config.HU_MAX)
    
    # Step 2: Apply CLAHE
    enhanced = apply_clahe(windowed, Config.CLAHE_CLIP_LIMIT)
    
    return enhanced


print("✅ Preprocessing functions defined!")
print("   Pipeline: Raw CT → HU Windowing [-75, 150] → CLAHE (clip=0.01)")

In [ ]:
# ============================================================================
# CELL 6: LOAD DATA AND EXTRACT 2D SLICES
# ============================================================================

def read_nii(filepath: str) -> np.ndarray:
    """Load NIfTI file and return as numpy array."""
    nii = nib.load(filepath)
    array = nii.get_fdata()
    array = np.rot90(np.array(array))  # Correct orientation
    return array.astype(np.float32)


def find_nii_files(data_paths: List[str]) -> Dict[str, List[Path]]:
    """Find all volume and segmentation NIfTI files."""
    volumes = []
    segmentations = []
    
    for data_path in data_paths:
        path = Path(data_path)
        for nii_file in path.rglob('*.nii*'):
            fname = nii_file.name.lower()
            if 'volume' in fname or 'liver' in fname:
                volumes.append(nii_file)
            elif 'segmentation' in fname or 'seg' in fname or 'label' in fname:
                segmentations.append(nii_file)
    
    return {'volumes': volumes, 'segmentations': segmentations}


def create_file_pairs(files: Dict) -> List[Dict]:
    """Match volume files with their corresponding segmentation files."""
    pairs = []
    
    for vol in files['volumes']:
        # Extract ID number from filename
        vol_name = vol.stem.replace('.nii', '')
        vol_id = ''.join(filter(str.isdigit, vol_name.split('-')[-1] if '-' in vol_name else vol_name.split('_')[-1]))
        
        # Find matching segmentation
        for seg in files['segmentations']:
            seg_name = seg.stem.replace('.nii', '')
            seg_id = ''.join(filter(str.isdigit, seg_name.split('-')[-1] if '-' in seg_name else seg_name.split('_')[-1]))
            
            if vol_id == seg_id and vol_id:
                pairs.append({'volume': vol, 'segmentation': seg, 'id': vol_id})
                break
    
    return pairs


# Find and pair files
print("📂 Searching for NIfTI files...")
files = find_nii_files(DATA_PATHS)
print(f"   Found {len(files['volumes'])} volumes")
print(f"   Found {len(files['segmentations'])} segmentations")

file_pairs = create_file_pairs(files)
print(f"\n✅ Created {len(file_pairs)} volume-segmentation pairs")

In [ ]:
# ============================================================================
# CELL 7: EXTRACT 2D SLICES WITH LABELS
# ============================================================================

def extract_slices_with_labels(file_pairs: List[Dict], max_cases: int = 10) -> Tuple[List, List]:
    """
    Extract 2D slices from 3D volumes with class labels for classification.
    
    Classification labels based on slice content:
    - Class 0 (Background): No liver visible
    - Class 1 (Liver): Liver present, no tumor
    - Class 2 (Tumor): Tumor present
    
    Args:
        file_pairs: List of volume-segmentation pairs
        max_cases: Maximum number of cases to process (for memory)
    
    Returns:
        (slices, labels) - preprocessed slices and their class labels
    """
    all_slices = []
    all_labels = []
    
    cases_to_process = min(len(file_pairs), max_cases)
    print(f"\n📊 Extracting slices from {cases_to_process} cases...\n")
    
    for i, pair in enumerate(tqdm(file_pairs[:cases_to_process], desc="Processing cases")):
        try:
            # Load volume and segmentation
            volume = read_nii(str(pair['volume']))
            segmentation = read_nii(str(pair['segmentation']))
            
            # Extract slices (skip edges with usually no content)
            n_slices = volume.shape[2]
            start = int(n_slices * 0.1)
            end = int(n_slices * 0.9)
            
            for s in range(start, end):
                ct_slice = volume[:, :, s]
                seg_slice = segmentation[:, :, s]
                
                # Determine class label based on segmentation content
                unique_labels = np.unique(seg_slice)
                
                if 2 in unique_labels:  # Tumor present
                    label = 2
                elif 1 in unique_labels:  # Liver only
                    label = 1
                else:  # Background
                    label = 0
                
                # Preprocess slice (HU Windowing + CLAHE)
                processed = preprocess_ct_slice(ct_slice)
                
                all_slices.append(processed)
                all_labels.append(label)
        
        except Exception as e:
            print(f"   ⚠️ Error processing case {pair['id']}: {e}")
            continue
    
    return all_slices, all_labels


# Extract slices
if file_pairs:
    slices, labels = extract_slices_with_labels(file_pairs, max_cases=5)  # Start with 5 cases
    
    # Convert to arrays
    slices = np.array(slices)
    labels = np.array(labels)
    
    print(f"\n✅ Dataset created:")
    print(f"   Total slices: {len(slices)}")
    print(f"   Shape: {slices.shape}")
    print(f"\n📊 Class distribution:")
    for i, name in enumerate(Config.CLASS_NAMES):
        count = (labels == i).sum()
        pct = count / len(labels) * 100
        print(f"   • {name}: {count} ({pct:.1f}%)")
else:
    print("❌ No file pairs found. Creating synthetic data...")
    # Synthetic fallback
    slices = np.random.rand(500, 512, 512).astype(np.float32)
    labels = np.random.randint(0, 3, 500)

In [ ]:
# ============================================================================
# CELL 8: VISUALIZE PREPROCESSING PIPELINE
# ============================================================================

def visualize_preprocessing(slices: np.ndarray, labels: np.ndarray, n_samples: int = 8):
    """
    Visualize samples from each class showing the preprocessing effect.
    """
    fig, axes = plt.subplots(3, n_samples // 3, figsize=(16, 12))
    axes = axes.flatten()
    
    # Get samples from each class
    samples = []
    for c in range(3):
        class_indices = np.where(labels == c)[0]
        if len(class_indices) > 0:
            n_per_class = min(n_samples // 3, len(class_indices))
            selected = np.random.choice(class_indices, n_per_class, replace=False)
            for idx in selected:
                samples.append((idx, c))
    
    # Class colors
    colors = {'Background': '#95a5a6', 'Liver': '#3498db', 'Tumor': '#e74c3c'}
    
    for i, (idx, c) in enumerate(samples[:len(axes)]):
        img = slices[idx]
        class_name = Config.CLASS_NAMES[c]
        
        axes[i].imshow(img, cmap='bone')
        axes[i].set_title(f"{class_name}", fontsize=12, 
                          color=colors[class_name], fontweight='bold')
        axes[i].axis('off')
    
    plt.suptitle('🔬 Preprocessed CT Slices by Class\n(HU Windowing + CLAHE Applied)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{Config.OUTPUT_DIR}/preprocessing_samples.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/preprocessing_samples.png")


# Visualize
visualize_preprocessing(slices, labels, n_samples=9)

In [ ]:
# ============================================================================
# CELL 9: PYTORCH DATASET WITH DATA AUGMENTATION
# ============================================================================

class LiverCTDataset(Dataset):
    """
    PyTorch Dataset for liver CT classification.
    
    Includes data augmentation for training:
    - Random horizontal flip
    - Random vertical flip  
    - Random rotation (up to 15 degrees)
    """
    
    def __init__(self, slices: np.ndarray, labels: np.ndarray, 
                 image_size: int = 224, is_train: bool = True):
        self.slices = slices
        self.labels = labels
        self.image_size = image_size
        self.is_train = is_train
        
        # Define transforms
        if is_train:
            self.transform = T.Compose([
                T.Resize((image_size, image_size)),
                T.RandomHorizontalFlip(p=0.5),
                T.RandomVerticalFlip(p=0.5),
                T.RandomRotation(degrees=15),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])  # ImageNet stats
            ])
        else:
            self.transform = T.Compose([
                T.Resize((image_size, image_size)),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
            ])
    
    def __len__(self):
        return len(self.slices)
    
    def __getitem__(self, idx):
        # Get slice and convert to 3-channel (for pretrained model)
        img = self.slices[idx]
        
        # Normalize to [0, 255] for PIL
        img_uint8 = (img * 255).astype(np.uint8)
        
        # Stack to 3 channels (grayscale → RGB-like)
        img_rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=2)
        
        # Convert to PIL for transforms
        pil_img = Image.fromarray(img_rgb)
        
        # Apply transforms
        tensor = self.transform(pil_img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return tensor, label


# Split data: Train / Validation / Test
print("📊 Splitting dataset...")

# First split: train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    slices, labels, test_size=Config.TEST_SPLIT, stratify=labels, random_state=42
)

# Second split: train vs val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=Config.VAL_SPLIT, stratify=y_trainval, random_state=42
)

print(f"   Train: {len(X_train)} samples")
print(f"   Val:   {len(X_val)} samples")
print(f"   Test:  {len(X_test)} samples")

# Create datasets
train_dataset = LiverCTDataset(X_train, y_train, Config.IMAGE_SIZE, is_train=True)
val_dataset = LiverCTDataset(X_val, y_val, Config.IMAGE_SIZE, is_train=False)
test_dataset = LiverCTDataset(X_test, y_test, Config.IMAGE_SIZE, is_train=False)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

print(f"\n✅ DataLoaders created!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

In [ ]:
# ============================================================================
# CELL 10: VISUALIZE DATA AUGMENTATION
# ============================================================================

def visualize_augmentation():
    """Show how data augmentation transforms the same image."""
    
    # Get a sample with tumor
    tumor_indices = np.where(y_train == 2)[0]
    if len(tumor_indices) == 0:
        tumor_indices = np.where(y_train == 1)[0]
    sample_idx = tumor_indices[0] if len(tumor_indices) > 0 else 0
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Original image
    original = X_train[sample_idx]
    axes[0, 0].imshow(original, cmap='bone')
    axes[0, 0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Show augmented versions
    aug_dataset = LiverCTDataset(X_train[sample_idx:sample_idx+1], 
                                  y_train[sample_idx:sample_idx+1], 
                                  Config.IMAGE_SIZE, is_train=True)
    
    for i in range(1, 8):
        row, col = divmod(i, 4)
        img_tensor, _ = aug_dataset[0]
        
        # Denormalize for visualization
        img = img_tensor.permute(1, 2, 0).numpy()
        img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)
        
        axes[row, col].imshow(img[:, :, 0], cmap='bone')
        axes[row, col].set_title(f'Augmented #{i}', fontsize=12)
        axes[row, col].axis('off')
    
    plt.suptitle('🔄 Data Augmentation Examples\n(Rotation + Flipping)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{Config.OUTPUT_DIR}/augmentation_examples.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/augmentation_examples.png")


visualize_augmentation()

In [ ]:
# ============================================================================
# CELL 11: MOBILENET-V1 CLASSIFIER MODEL
# ============================================================================

class MobileNetClassifier(nn.Module):
    """
    MobileNet-V1 based classifier for liver CT classification.
    
    Uses transfer learning from ImageNet pretrained weights.
    The classifier head is replaced for 3-class classification.
    """
    
    def __init__(self, num_classes: int = 3, pretrained: bool = True):
        super().__init__()
        
        # Load pretrained MobileNet-V2 (V1 not directly available in torchvision)
        # MobileNet-V2 is similar but more efficient
        weights = 'IMAGENET1K_V1' if pretrained else None
        self.backbone = models.mobilenet_v2(weights=weights)
        
        # Get the number of input features for the classifier
        in_features = self.backbone.classifier[1].in_features
        
        # Replace classifier head
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )
        
        # Count parameters
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        print(f"🧠 MobileNet-V2 Classifier:")
        print(f"   Total parameters: {total_params:,}")
        print(f"   Trainable parameters: {trainable_params:,}")
        print(f"   Output classes: {num_classes}")
    
    def forward(self, x):
        return self.backbone(x)


# Create model
model = MobileNetClassifier(num_classes=Config.NUM_CLASSES, pretrained=True).to(DEVICE)

# Test forward pass
test_input = torch.randn(1, 3, Config.IMAGE_SIZE, Config.IMAGE_SIZE).to(DEVICE)
test_output = model(test_input)
print(f"\n✅ Test forward pass: Input {tuple(test_input.shape)} → Output {tuple(test_output.shape)}")

In [ ]:
# ============================================================================
# CELL 12: EVALUATION METRICS (DSC & Sensitivity)
# ============================================================================

def dice_coefficient(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = 3) -> Dict[str, float]:
    """
    Calculate Dice Similarity Coefficient (DSC) per class.
    
    DSC = 2 * |X ∩ Y| / (|X| + |Y|)
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
        num_classes: Number of classes
    
    Returns:
        Dictionary with per-class DSC and mean DSC
    """
    dice_scores = {}
    
    for c in range(num_classes):
        true_mask = (y_true == c)
        pred_mask = (y_pred == c)
        
        intersection = (true_mask & pred_mask).sum()
        union = true_mask.sum() + pred_mask.sum()
        
        if union == 0:
            dice = 1.0  # Both empty
        else:
            dice = 2 * intersection / union
        
        dice_scores[Config.CLASS_NAMES[c]] = dice
    
    # Mean DSC (excluding background)
    dice_scores['Mean (Liver+Tumor)'] = np.mean([dice_scores['Liver'], dice_scores['Tumor']])
    
    return dice_scores


def sensitivity(y_true: np.ndarray, y_pred: np.ndarray, positive_class: int = 2) -> float:
    """
    Calculate Sensitivity (True Positive Rate / Recall) for tumor detection.
    
    Sensitivity = TP / (TP + FN)
    
    This metric is crucial for cancer detection - we want to minimize false negatives.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels  
        positive_class: Class to calculate sensitivity for (default: 2=Tumor)
    
    Returns:
        Sensitivity score
    """
    true_positive = ((y_true == positive_class) & (y_pred == positive_class)).sum()
    false_negative = ((y_true == positive_class) & (y_pred != positive_class)).sum()
    
    if (true_positive + false_negative) == 0:
        return 1.0  # No positive samples
    
    return true_positive / (true_positive + false_negative)


def specificity(y_true: np.ndarray, y_pred: np.ndarray, positive_class: int = 2) -> float:
    """
    Calculate Specificity (True Negative Rate).
    
    Specificity = TN / (TN + FP)
    """
    true_negative = ((y_true != positive_class) & (y_pred != positive_class)).sum()
    false_positive = ((y_true != positive_class) & (y_pred == positive_class)).sum()
    
    if (true_negative + false_positive) == 0:
        return 1.0
    
    return true_negative / (true_negative + false_positive)


print("✅ Evaluation metrics defined!")
print("   • Dice Similarity Coefficient (DSC)")
print("   • Sensitivity (Recall)")
print("   • Specificity")

In [ ]:
# ============================================================================
# CELL 13: TRAINING LOOP
# ============================================================================

def train_model(model, train_loader, val_loader, epochs: int, lr: float):
    """
    Train the MobileNet classifier.
    
    Uses:
    - Adam optimizer with specified learning rate
    - CrossEntropy loss for multi-class classification
    - Learning rate scheduling
    """
    # Loss and optimizer (Adam with lr=0.001 as specified)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    # History tracking
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'val_sensitivity': [], 'val_dice': []
    }
    
    best_val_acc = 0.0
    
    print("\n" + "="*70)
    print("🚀 TRAINING MOBILENET-V2 CLASSIFIER")
    print("="*70)
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train Acc':>10} | {'Val Acc':>10} | {'Sensitivity':>11}")
    print("-"*70)
    
    for epoch in range(1, epochs + 1):
        # ====== TRAINING ======
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
        
        train_loss /= len(train_loader.dataset)
        train_acc = train_correct / len(train_loader.dataset)
        
        # ====== VALIDATION ======
        model.eval()
        val_loss, val_correct = 0.0, 0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                
                all_preds.extend(outputs.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss /= len(val_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)
        
        # Calculate metrics
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        sens = sensitivity(all_labels, all_preds)
        dice = dice_coefficient(all_labels, all_preds)
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Save best model
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{Config.MODEL_DIR}/mobilenet_best.pth")
        
        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_sensitivity'].append(sens)
        history['val_dice'].append(dice['Mean (Liver+Tumor)'])
        
        # Print progress
        star = '⭐' if is_best else ''
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {train_acc:>10.4f} | {val_acc:>10.4f} | {sens:>11.4f} {star}")
    
    print("-"*70)
    print(f"✅ Training complete! Best validation accuracy: {best_val_acc:.4f}")
    
    return history


# Train the model
history = train_model(
    model, train_loader, val_loader,
    epochs=Config.EPOCHS,
    lr=Config.LEARNING_RATE
)

In [ ]:
# ============================================================================
# CELL 14: TRAINING HISTORY VISUALIZATION
# ============================================================================

def plot_training_history(history: Dict):
    """Plot training metrics over epochs."""
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train')
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Validation')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('📉 Loss Curve', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', linewidth=2, label='Train')
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', linewidth=2, label='Validation')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('📈 Accuracy Curve', fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 1.05])
    
    # Sensitivity
    axes[1, 0].plot(epochs, history['val_sensitivity'], 'g-', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Sensitivity')
    axes[1, 0].set_title('🎯 Tumor Sensitivity', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_ylim([0, 1.05])
    
    # Dice Score
    axes[1, 1].plot(epochs, history['val_dice'], 'm-', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Dice Score')
    axes[1, 1].set_title('🎲 Mean Dice Coefficient', fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_ylim([0, 1.05])
    
    plt.suptitle('📊 Training History', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{Config.OUTPUT_DIR}/training_history.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/training_history.png")


plot_training_history(history)

In [ ]:
# ============================================================================
# CELL 15: TEST SET EVALUATION
# ============================================================================

@torch.no_grad()
def evaluate_model(model, test_loader):
    """
    Evaluate model on test set.
    
    Returns predictions and ground truth for detailed analysis.
    """
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(DEVICE)
        
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        preds = outputs.argmax(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


# Load best model
print("📂 Loading best model...")
try:
    model.load_state_dict(torch.load(f"{Config.MODEL_DIR}/mobilenet_best.pth"))
    print("✅ Best model loaded!")
except:
    print("⚠️ Using current model weights")

# Evaluate on test set
print("\n🧪 Evaluating on test set...")
test_preds, test_labels, test_probs = evaluate_model(model, test_loader)

# Calculate accuracy
test_accuracy = (test_preds == test_labels).mean()
print(f"\n📊 Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# ============================================================================
# CELL 16: CONFUSION MATRIX
# ============================================================================

def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str]):
    """
    Plot a detailed confusion matrix with percentages.
    """
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], cbar_kws={'label': 'Count'})
    axes[0].set_xlabel('Predicted', fontsize=12)
    axes[0].set_ylabel('Actual', fontsize=12)
    axes[0].set_title('📊 Confusion Matrix (Counts)', fontweight='bold')
    
    # Normalized (percentages)
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], cbar_kws={'label': 'Percentage'})
    axes[1].set_xlabel('Predicted', fontsize=12)
    axes[1].set_ylabel('Actual', fontsize=12)
    axes[1].set_title('📊 Confusion Matrix (Normalized)', fontweight='bold')
    
    plt.suptitle('🔬 Test Set Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{Config.OUTPUT_DIR}/confusion_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/confusion_matrix.png")
    
    return cm


# Plot confusion matrix
cm = plot_confusion_matrix(test_labels, test_preds, Config.CLASS_NAMES)

In [ ]:
# ============================================================================
# CELL 17: DETAILED METRICS REPORT
# ============================================================================

def print_detailed_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Print comprehensive evaluation metrics.
    """
    print("\n" + "="*70)
    print("📊 DETAILED EVALUATION METRICS")
    print("="*70)
    
    # Scikit-learn classification report
    print("\n📋 Classification Report:")
    print("-"*70)
    print(classification_report(y_true, y_pred, target_names=Config.CLASS_NAMES, digits=4))
    
    # Dice Scores
    print("\n🎲 Dice Similarity Coefficient (DSC):")
    print("-"*70)
    dice = dice_coefficient(y_true, y_pred)
    for name, score in dice.items():
        print(f"   {name:20s}: {score:.4f}")
    
    # Sensitivity and Specificity
    print("\n🎯 Cancer Detection Metrics:")
    print("-"*70)
    sens = sensitivity(y_true, y_pred, positive_class=2)
    spec = specificity(y_true, y_pred, positive_class=2)
    print(f"   Tumor Sensitivity (Recall): {sens:.4f}")
    print(f"   Tumor Specificity:          {spec:.4f}")
    
    # Per-class sensitivity
    print("\n   Per-class Sensitivity:")
    for c, name in enumerate(Config.CLASS_NAMES):
        sens_c = sensitivity(y_true, y_pred, positive_class=c)
        print(f"      {name}: {sens_c:.4f}")
    
    print("\n" + "="*70)
    
    return {
        'dice': dice,
        'sensitivity': sens,
        'specificity': spec,
        'accuracy': (y_true == y_pred).mean()
    }


# Print detailed metrics
metrics = print_detailed_metrics(test_labels, test_preds)

In [ ]:
# ============================================================================
# CELL 18: ENHANCED CANCER DETECTION VISUALIZATION
# ============================================================================

def visualize_predictions_enhanced(model, test_loader, n_samples: int = 12):
    """
    Enhanced visualization with tumor highlighting, probability heatmaps,
    and detection overlays similar to clinical imaging software.
    """
    model.eval()
    
    # Collect samples
    images_list, labels_list, preds_list, probs_list = [], [], [], []
    
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(DEVICE))
            probs = F.softmax(outputs, dim=1)
            preds = outputs.argmax(1)
            
            images_list.extend(images.numpy())
            labels_list.extend(labels.numpy())
            preds_list.extend(preds.cpu().numpy())
            probs_list.extend(probs.cpu().numpy())
            
            if len(images_list) >= n_samples:
                break
    
    # Create enhanced visualization
    n = min(n_samples, len(images_list))
    fig, axes = plt.subplots(n, 4, figsize=(20, 5*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    
    # Color maps for classes
    class_colors = {
        0: '#95a5a6',  # Background - Gray
        1: '#3498db',  # Liver - Blue
        2: '#e74c3c'   # Tumor - Red
    }
    
    for i in range(n):
        # Denormalize image
        img = images_list[i].transpose(1, 2, 0)
        img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)
        img_gray = img[:, :, 0]
        
        true_label = labels_list[i]
        pred_label = preds_list[i]
        probs = probs_list[i]
        
        is_correct = true_label == pred_label
        
        # Column 1: Original CT slice
        axes[i, 0].imshow(img_gray, cmap='bone')
        axes[i, 0].set_title('📷 CT Slice (Preprocessed)', fontsize=11, fontweight='bold')
        axes[i, 0].axis('off')
        
        # Column 2: Classification result with color coding
        axes[i, 1].imshow(img_gray, cmap='bone')
        
        # Add classification box
        color = class_colors[pred_label]
        symbol = '✓' if is_correct else '✗'
        result_color = 'green' if is_correct else 'red'
        
        # Add border effect
        for spine in axes[i, 1].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        
        axes[i, 1].set_title(f'{symbol} Pred: {Config.CLASS_NAMES[pred_label]}\nTrue: {Config.CLASS_NAMES[true_label]}', 
                             fontsize=11, fontweight='bold', color=result_color)
        axes[i, 1].axis('off')
        
        # Column 3: Probability distribution bar chart
        bars = axes[i, 2].barh(Config.CLASS_NAMES, probs * 100, 
                                color=[class_colors[j] for j in range(3)])
        axes[i, 2].set_xlim([0, 100])
        axes[i, 2].set_xlabel('Confidence (%)', fontsize=10)
        axes[i, 2].set_title('📊 Class Probabilities', fontsize=11, fontweight='bold')
        
        # Add percentage labels
        for bar, prob in zip(bars, probs):
            width = bar.get_width()
            axes[i, 2].text(width + 2, bar.get_y() + bar.get_height()/2, 
                           f'{prob*100:.1f}%', va='center', fontsize=9, fontweight='bold')
        
        # Highlight predicted class
        bars[pred_label].set_edgecolor('black')
        bars[pred_label].set_linewidth(2)
        
        # Column 4: Cancer detection overlay
        axes[i, 3].imshow(img_gray, cmap='bone')
        
        if pred_label == 2:  # Tumor predicted
            # Create red overlay for tumor indication
            red_overlay = np.zeros((*img_gray.shape, 4))
            red_overlay[:, :, 0] = 1.0  # Red channel
            red_overlay[:, :, 3] = 0.3  # Alpha
            axes[i, 3].imshow(red_overlay)
            
            # Add warning indicator
            axes[i, 3].text(0.5, 0.95, '⚠️ TUMOR DETECTED', transform=axes[i, 3].transAxes,
                           ha='center', va='top', fontsize=12, fontweight='bold',
                           color='white', bbox=dict(boxstyle='round', facecolor='#e74c3c', alpha=0.9))
            axes[i, 3].set_title(f'🔴 Cancer Detection: {probs[2]*100:.1f}%', 
                                fontsize=11, fontweight='bold', color='#e74c3c')
        elif pred_label == 1:  # Liver only
            # Create blue overlay for liver
            blue_overlay = np.zeros((*img_gray.shape, 4))
            blue_overlay[:, :, 2] = 1.0  # Blue channel
            blue_overlay[:, :, 3] = 0.2  # Alpha
            axes[i, 3].imshow(blue_overlay)
            
            axes[i, 3].text(0.5, 0.95, '✓ LIVER (No Tumor)', transform=axes[i, 3].transAxes,
                           ha='center', va='top', fontsize=12, fontweight='bold',
                           color='white', bbox=dict(boxstyle='round', facecolor='#3498db', alpha=0.9))
            axes[i, 3].set_title(f'🔵 Liver Detection: {probs[1]*100:.1f}%', 
                                fontsize=11, fontweight='bold', color='#3498db')
        else:  # Background
            axes[i, 3].text(0.5, 0.95, '— Background', transform=axes[i, 3].transAxes,
                           ha='center', va='top', fontsize=12, fontweight='bold',
                           color='white', bbox=dict(boxstyle='round', facecolor='#95a5a6', alpha=0.9))
            axes[i, 3].set_title('⬜ No Liver/Tumor', fontsize=11, fontweight='bold', color='#95a5a6')
        
        axes[i, 3].axis('off')
    
    plt.suptitle('🔬 HEPATOCELLULAR CARCINOMA DETECTION RESULTS\nCT Slice → Classification → Probability → Cancer Overlay', 
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f"{Config.OUTPUT_DIR}/cancer_detection_enhanced.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/cancer_detection_enhanced.png")


# Run enhanced visualization
visualize_predictions_enhanced(model, test_loader, n_samples=6)

In [ ]:
# ============================================================================
# CELL 18B: CLINICAL CANCER DETECTION DASHBOARD
# ============================================================================

def create_cancer_detection_dashboard(test_preds, test_labels, test_probs):
    """
    Create a comprehensive clinical-style dashboard for cancer detection results.
    Similar to medical imaging software displays.
    """
    fig = plt.figure(figsize=(20, 14))
    
    # Create grid layout
    gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)
    
    # ===== ROW 1: Key Metrics =====
    
    # Calculate key statistics
    total_samples = len(test_labels)
    tumor_true = (test_labels == 2).sum()
    tumor_pred = (test_preds == 2).sum()
    tumor_correct = ((test_labels == 2) & (test_preds == 2)).sum()
    
    liver_true = (test_labels == 1).sum()
    liver_pred = (test_preds == 1).sum()
    liver_correct = ((test_labels == 1) & (test_preds == 1)).sum()
    
    # Tumor detection rate (sensitivity)
    tumor_sensitivity = tumor_correct / tumor_true * 100 if tumor_true > 0 else 0
    tumor_precision = tumor_correct / tumor_pred * 100 if tumor_pred > 0 else 0
    
    # Panel 1: Large tumor detection indicator
    ax1 = fig.add_subplot(gs[0, 0:2])
    ax1.set_facecolor('#1a1a2e')
    
    # Determine overall status
    if tumor_sensitivity >= 90:
        status_color = '#27ae60'
        status_text = 'EXCELLENT'
        status_emoji = '🟢'
    elif tumor_sensitivity >= 75:
        status_color = '#f39c12'
        status_text = 'GOOD'
        status_emoji = '🟡'
    else:
        status_color = '#e74c3c'
        status_text = 'NEEDS IMPROVEMENT'
        status_emoji = '🔴'
    
    ax1.text(0.5, 0.7, f'{status_emoji} TUMOR DETECTION: {status_text}', 
             transform=ax1.transAxes, ha='center', va='center',
             fontsize=20, fontweight='bold', color='white')
    ax1.text(0.5, 0.35, f'Sensitivity: {tumor_sensitivity:.1f}%', 
             transform=ax1.transAxes, ha='center', va='center',
             fontsize=28, fontweight='bold', color=status_color)
    ax1.text(0.5, 0.1, f'{tumor_correct}/{tumor_true} tumors correctly identified', 
             transform=ax1.transAxes, ha='center', va='center',
             fontsize=14, color='#bdc3c7')
    ax1.axis('off')
    
    # Panel 2: Detection statistics
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax2.set_facecolor('#1a1a2e')
    
    stats_text = f"""
    📊 DETECTION SUMMARY
    ─────────────────────────
    Total Samples:     {total_samples}
    
    🔴 Tumor Cases:    {tumor_true} true / {tumor_pred} predicted
       Sensitivity:    {tumor_sensitivity:.1f}%
       Precision:      {tumor_precision:.1f}%
    
    🔵 Liver Cases:    {liver_true} true / {liver_pred} predicted
       Accuracy:       {liver_correct/liver_true*100 if liver_true > 0 else 0:.1f}%
    
    ⬜ Background:     {(test_labels == 0).sum()} samples
    """
    ax2.text(0.05, 0.5, stats_text, transform=ax2.transAxes, 
             fontsize=12, fontfamily='monospace', color='white', va='center')
    ax2.axis('off')
    
    # ===== ROW 2: Visualizations =====
    
    # Panel 3: Pie chart of predictions
    ax3 = fig.add_subplot(gs[1, 0])
    pred_counts = [(test_preds == i).sum() for i in range(3)]
    colors = ['#95a5a6', '#3498db', '#e74c3c']
    explode = (0, 0, 0.1)  # Highlight tumor
    wedges, texts, autotexts = ax3.pie(pred_counts, labels=Config.CLASS_NAMES, 
                                        colors=colors, explode=explode,
                                        autopct='%1.1f%%', startangle=90,
                                        textprops={'fontsize': 10, 'fontweight': 'bold'})
    ax3.set_title('🎯 Prediction Distribution', fontsize=12, fontweight='bold')
    
    # Panel 4: Confusion matrix heatmap
    ax4 = fig.add_subplot(gs[1, 1:3])
    cm = confusion_matrix(test_labels, test_preds)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(cm_normalized, annot=True, fmt='.1%', cmap='RdYlGn_r',
                xticklabels=Config.CLASS_NAMES, yticklabels=Config.CLASS_NAMES,
                ax=ax4, cbar_kws={'label': 'Rate'}, vmin=0, vmax=1,
                annot_kws={'fontsize': 12, 'fontweight': 'bold'})
    ax4.set_xlabel('Predicted', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Actual', fontsize=11, fontweight='bold')
    ax4.set_title('🔬 Classification Performance Matrix', fontsize=12, fontweight='bold')
    
    # Panel 5: Tumor probability histogram
    ax5 = fig.add_subplot(gs[1, 3])
    tumor_probs = test_probs[:, 2]  # Tumor class probabilities
    
    # Separate by true label
    tumor_probs_positive = tumor_probs[test_labels == 2]
    tumor_probs_negative = tumor_probs[test_labels != 2]
    
    ax5.hist(tumor_probs_negative, bins=20, alpha=0.7, color='#3498db', 
             label='Non-tumor', edgecolor='black')
    ax5.hist(tumor_probs_positive, bins=20, alpha=0.7, color='#e74c3c', 
             label='Tumor', edgecolor='black')
    ax5.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold')
    ax5.set_xlabel('Tumor Probability', fontsize=10)
    ax5.set_ylabel('Count', fontsize=10)
    ax5.set_title('📈 Tumor Probability Distribution', fontsize=12, fontweight='bold')
    ax5.legend(fontsize=9)
    
    # ===== ROW 3: Per-class metrics =====
    
    # Panel 6: Per-class sensitivity bars
    ax6 = fig.add_subplot(gs[2, 0:2])
    
    sensitivities = []
    specificities = []
    for c in range(3):
        sens = sensitivity(test_labels, test_preds, positive_class=c) * 100
        spec = specificity(test_labels, test_preds, positive_class=c) * 100
        sensitivities.append(sens)
        specificities.append(spec)
    
    x = np.arange(3)
    width = 0.35
    bars1 = ax6.bar(x - width/2, sensitivities, width, label='Sensitivity', color='#2ecc71')
    bars2 = ax6.bar(x + width/2, specificities, width, label='Specificity', color='#9b59b6')
    
    ax6.set_ylabel('Percentage (%)', fontsize=11)
    ax6.set_title('🎯 Per-Class Detection Metrics', fontsize=12, fontweight='bold')
    ax6.set_xticks(x)
    ax6.set_xticklabels(Config.CLASS_NAMES, fontsize=10, fontweight='bold')
    ax6.legend(fontsize=10)
    ax6.set_ylim([0, 110])
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Panel 7: Dice scores
    ax7 = fig.add_subplot(gs[2, 2:4])
    
    dice_scores = dice_coefficient(test_labels, test_preds)
    dice_names = list(dice_scores.keys())
    dice_values = [v * 100 for v in dice_scores.values()]
    colors_dice = ['#95a5a6', '#3498db', '#e74c3c', '#f39c12']
    
    bars = ax7.barh(dice_names, dice_values, color=colors_dice, edgecolor='black')
    ax7.set_xlim([0, 110])
    ax7.set_xlabel('Dice Score (%)', fontsize=11)
    ax7.set_title('🎲 Dice Similarity Coefficients', fontsize=12, fontweight='bold')
    
    for bar, val in zip(bars, dice_values):
        ax7.text(val + 2, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
    
    plt.suptitle('🏥 HEPATOCELLULAR CARCINOMA DETECTION - CLINICAL DASHBOARD', 
                 fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(f"{Config.OUTPUT_DIR}/clinical_dashboard.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/clinical_dashboard.png")


# Create the clinical dashboard
create_cancer_detection_dashboard(test_preds, test_labels, test_probs)

In [ ]:
# ============================================================================
# CELL 18C: DETAILED TUMOR CASE ANALYSIS
# ============================================================================

def analyze_tumor_cases(model, test_loader, test_preds, test_labels, test_probs):
    """
    Detailed analysis of tumor detection cases with visual highlighting.
    Shows correctly detected tumors, missed tumors, and false positives.
    """
    model.eval()
    
    # Collect all test images
    images_list = []
    with torch.no_grad():
        for images, _ in test_loader:
            images_list.extend(images.numpy())
    images_list = np.array(images_list)
    
    # Find different case types
    true_positive_idx = np.where((test_labels == 2) & (test_preds == 2))[0]  # Correctly detected tumors
    false_negative_idx = np.where((test_labels == 2) & (test_preds != 2))[0]  # Missed tumors
    false_positive_idx = np.where((test_labels != 2) & (test_preds == 2))[0]  # False alarms
    
    fig = plt.figure(figsize=(20, 16))
    
    # ===== Section 1: Correctly Detected Tumors (TRUE POSITIVES) =====
    n_tp = min(4, len(true_positive_idx))
    for i in range(n_tp):
        ax = fig.add_subplot(4, 4, i + 1)
        idx = true_positive_idx[i]
        
        img = images_list[idx].transpose(1, 2, 0)
        img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)
        
        ax.imshow(img[:, :, 0], cmap='bone')
        
        # Red glow effect for tumor
        red_overlay = np.zeros((*img[:,:,0].shape, 4))
        red_overlay[:, :, 0] = 1.0
        red_overlay[:, :, 3] = 0.25
        ax.imshow(red_overlay)
        
        # Add border and label
        for spine in ax.spines.values():
            spine.set_edgecolor('#27ae60')
            spine.set_linewidth(4)
        
        conf = test_probs[idx, 2] * 100
        ax.set_title(f'✓ DETECTED ({conf:.1f}%)', fontsize=10, 
                    fontweight='bold', color='#27ae60')
        ax.axis('off')
    
    # Fill remaining slots with placeholder
    for i in range(n_tp, 4):
        ax = fig.add_subplot(4, 4, i + 1)
        ax.text(0.5, 0.5, 'No more\nsamples', ha='center', va='center', 
               fontsize=12, color='#95a5a6')
        ax.set_facecolor('#f0f0f0')
        ax.axis('off')
    
    # Add section header
    fig.text(0.25, 0.95, '✅ TRUE POSITIVES (Correctly Detected Tumors)', 
             fontsize=14, fontweight='bold', ha='center', color='#27ae60')
    
    # ===== Section 2: Missed Tumors (FALSE NEGATIVES) =====
    n_fn = min(4, len(false_negative_idx))
    for i in range(n_fn):
        ax = fig.add_subplot(4, 4, i + 5)
        idx = false_negative_idx[i]
        
        img = images_list[idx].transpose(1, 2, 0)
        img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)
        
        ax.imshow(img[:, :, 0], cmap='bone')
        
        # Yellow warning overlay
        yellow_overlay = np.zeros((*img[:,:,0].shape, 4))
        yellow_overlay[:, :, 0] = 1.0
        yellow_overlay[:, :, 1] = 0.8
        yellow_overlay[:, :, 3] = 0.2
        ax.imshow(yellow_overlay)
        
        for spine in ax.spines.values():
            spine.set_edgecolor('#e74c3c')
            spine.set_linewidth(4)
        
        pred_class = Config.CLASS_NAMES[test_preds[idx]]
        ax.set_title(f'✗ MISSED (Pred: {pred_class})', fontsize=10, 
                    fontweight='bold', color='#e74c3c')
        ax.axis('off')
    
    for i in range(n_fn, 4):
        ax = fig.add_subplot(4, 4, i + 5)
        ax.text(0.5, 0.5, 'No missed\ntumors! 🎉', ha='center', va='center', 
               fontsize=12, color='#27ae60')
        ax.set_facecolor('#e8f5e9')
        ax.axis('off')
    
    fig.text(0.75, 0.95, '❌ FALSE NEGATIVES (Missed Tumors)', 
             fontsize=14, fontweight='bold', ha='center', color='#e74c3c')
    
    # ===== Section 3: False Positives =====
    n_fp = min(4, len(false_positive_idx))
    for i in range(n_fp):
        ax = fig.add_subplot(4, 4, i + 9)
        idx = false_positive_idx[i]
        
        img = images_list[idx].transpose(1, 2, 0)
        img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)
        
        ax.imshow(img[:, :, 0], cmap='bone')
        
        # Orange overlay for false alarm
        orange_overlay = np.zeros((*img[:,:,0].shape, 4))
        orange_overlay[:, :, 0] = 1.0
        orange_overlay[:, :, 1] = 0.5
        orange_overlay[:, :, 3] = 0.2
        ax.imshow(orange_overlay)
        
        for spine in ax.spines.values():
            spine.set_edgecolor('#f39c12')
            spine.set_linewidth(4)
        
        true_class = Config.CLASS_NAMES[test_labels[idx]]
        ax.set_title(f'⚠ FALSE ALARM (True: {true_class})', fontsize=10, 
                    fontweight='bold', color='#f39c12')
        ax.axis('off')
    
    for i in range(n_fp, 4):
        ax = fig.add_subplot(4, 4, i + 9)
        ax.text(0.5, 0.5, 'No false\nalarms! 🎉', ha='center', va='center', 
               fontsize=12, color='#27ae60')
        ax.set_facecolor('#e8f5e9')
        ax.axis('off')
    
    fig.text(0.25, 0.52, '⚠️ FALSE POSITIVES (False Tumor Alarms)', 
             fontsize=14, fontweight='bold', ha='center', color='#f39c12')
    
    # ===== Section 4: Summary Statistics =====
    ax_summary = fig.add_subplot(4, 4, 13)
    ax_summary.set_facecolor('#1a1a2e')
    
    summary_text = f"""
    📊 TUMOR DETECTION
    ────────────────
    ✅ True Pos:  {len(true_positive_idx)}
    ❌ False Neg: {len(false_negative_idx)}
    ⚠️  False Pos: {len(false_positive_idx)}
    """
    ax_summary.text(0.1, 0.5, summary_text, transform=ax_summary.transAxes,
                   fontsize=11, fontfamily='monospace', color='white', va='center')
    ax_summary.axis('off')
    
    # Sensitivity gauge
    ax_gauge = fig.add_subplot(4, 4, 14)
    tumor_sens = len(true_positive_idx) / (len(true_positive_idx) + len(false_negative_idx)) * 100 if (len(true_positive_idx) + len(false_negative_idx)) > 0 else 0
    
    # Create semi-circle gauge
    theta = np.linspace(0, np.pi, 100)
    r = 1
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    
    ax_gauge.fill_between(x, 0, y, color='#ecf0f1')
    
    # Fill based on sensitivity
    fill_angle = np.pi * tumor_sens / 100
    x_fill = r * np.cos(np.linspace(0, fill_angle, 50))
    y_fill = r * np.sin(np.linspace(0, fill_angle, 50))
    
    color = '#27ae60' if tumor_sens >= 80 else '#f39c12' if tumor_sens >= 60 else '#e74c3c'
    ax_gauge.fill_between(x_fill, 0, y_fill, color=color, alpha=0.8)
    
    ax_gauge.text(0, -0.3, f'{tumor_sens:.1f}%', ha='center', fontsize=20, fontweight='bold')
    ax_gauge.text(0, -0.5, 'Sensitivity', ha='center', fontsize=12)
    ax_gauge.set_xlim([-1.2, 1.2])
    ax_gauge.set_ylim([-0.6, 1.2])
    ax_gauge.set_aspect('equal')
    ax_gauge.axis('off')
    
    # Precision gauge
    ax_prec = fig.add_subplot(4, 4, 15)
    tumor_prec = len(true_positive_idx) / (len(true_positive_idx) + len(false_positive_idx)) * 100 if (len(true_positive_idx) + len(false_positive_idx)) > 0 else 0
    
    ax_prec.fill_between(x, 0, y, color='#ecf0f1')
    
    fill_angle = np.pi * tumor_prec / 100
    x_fill = r * np.cos(np.linspace(0, fill_angle, 50))
    y_fill = r * np.sin(np.linspace(0, fill_angle, 50))
    
    color = '#27ae60' if tumor_prec >= 80 else '#f39c12' if tumor_prec >= 60 else '#e74c3c'
    ax_prec.fill_between(x_fill, 0, y_fill, color=color, alpha=0.8)
    
    ax_prec.text(0, -0.3, f'{tumor_prec:.1f}%', ha='center', fontsize=20, fontweight='bold')
    ax_prec.text(0, -0.5, 'Precision', ha='center', fontsize=12)
    ax_prec.set_xlim([-1.2, 1.2])
    ax_prec.set_ylim([-0.6, 1.2])
    ax_prec.set_aspect('equal')
    ax_prec.axis('off')
    
    # F1 Score
    ax_f1 = fig.add_subplot(4, 4, 16)
    f1_score = 2 * tumor_sens * tumor_prec / (tumor_sens + tumor_prec) if (tumor_sens + tumor_prec) > 0 else 0
    
    ax_f1.fill_between(x, 0, y, color='#ecf0f1')
    
    fill_angle = np.pi * f1_score / 100
    x_fill = r * np.cos(np.linspace(0, fill_angle, 50))
    y_fill = r * np.sin(np.linspace(0, fill_angle, 50))
    
    color = '#27ae60' if f1_score >= 80 else '#f39c12' if f1_score >= 60 else '#e74c3c'
    ax_f1.fill_between(x_fill, 0, y_fill, color=color, alpha=0.8)
    
    ax_f1.text(0, -0.3, f'{f1_score:.1f}%', ha='center', fontsize=20, fontweight='bold')
    ax_f1.text(0, -0.5, 'F1 Score', ha='center', fontsize=12)
    ax_f1.set_xlim([-1.2, 1.2])
    ax_f1.set_ylim([-0.6, 1.2])
    ax_f1.set_aspect('equal')
    ax_f1.axis('off')
    
    fig.text(0.75, 0.52, '📈 DETECTION METRICS', 
             fontsize=14, fontweight='bold', ha='center', color='#2c3e50')
    
    plt.suptitle('🔬 TUMOR DETECTION CASE ANALYSIS\nTrue Positives | False Negatives | False Positives | Metrics', 
                 fontsize=16, fontweight='bold', y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"{Config.OUTPUT_DIR}/tumor_case_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/tumor_case_analysis.png")


# Run tumor case analysis
analyze_tumor_cases(model, test_loader, test_preds, test_labels, test_probs)

In [ ]:
# ============================================================================
# CELL 19: CLASS DISTRIBUTION VISUALIZATION
# ============================================================================

def plot_class_distribution():
    """Visualize class distribution across train/val/test splits."""
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    datasets = [
        ('Train', y_train),
        ('Validation', y_val),
        ('Test', y_test)
    ]
    
    colors = ['#3498db', '#2ecc71', '#e74c3c']
    
    for ax, (name, data) in zip(axes, datasets):
        counts = [np.sum(data == i) for i in range(3)]
        bars = ax.bar(Config.CLASS_NAMES, counts, color=colors)
        ax.set_title(f'{name} Set', fontsize=12, fontweight='bold')
        ax.set_ylabel('Count')
        
        # Add count labels
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                   str(count), ha='center', fontweight='bold')
    
    plt.suptitle('📊 Class Distribution Across Splits', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{Config.OUTPUT_DIR}/class_distribution.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved to {Config.OUTPUT_DIR}/class_distribution.png")


plot_class_distribution()

In [ ]:
# ============================================================================
# CELL 20: FINAL SUMMARY
# ============================================================================

from IPython.display import HTML

# Calculate final metrics
final_acc = (test_preds == test_labels).mean() * 100
final_sens = sensitivity(test_labels, test_preds) * 100
final_dice = dice_coefficient(test_labels, test_preds)['Mean (Liver+Tumor)'] * 100

html_summary = f"""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); padding: 30px; border-radius: 15px; color: white; font-family: Arial;">
    
    <h1 style="text-align: center; color: #00d4ff;">🎯 LIVER CANCER CLASSIFICATION - FINAL RESULTS</h1>
    
    <hr style="border-color: #00d4ff;">
    
    <h2 style="color: #3498db;">📋 Model Configuration</h2>
    <table style="width: 100%; border-collapse: collapse;">
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #444;">🧠 Architecture</td>
            <td style="padding: 10px; border-bottom: 1px solid #444;"><b>MobileNet-V2 (Transfer Learning)</b></td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #444;">📐 Input Size</td>
            <td style="padding: 10px; border-bottom: 1px solid #444;">{Config.IMAGE_SIZE} x {Config.IMAGE_SIZE}</td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #444;">🎯 Preprocessing</td>
            <td style="padding: 10px; border-bottom: 1px solid #444;">HU Window [{Config.HU_MIN}, {Config.HU_MAX}] + CLAHE (clip={Config.CLAHE_CLIP_LIMIT})</td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #444;">🔄 Augmentation</td>
            <td style="padding: 10px; border-bottom: 1px solid #444;">Random Flip + Rotation (±15°)</td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #444;">⚙️ Optimizer</td>
            <td style="padding: 10px; border-bottom: 1px solid #444;">Adam (lr={Config.LEARNING_RATE})</td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #444;">📊 Epochs</td>
            <td style="padding: 10px; border-bottom: 1px solid #444;">{Config.EPOCHS}</td>
        </tr>
    </table>
    
    <h2 style="color: #2ecc71; margin-top: 30px;">📈 Performance Metrics</h2>
    
    <div style="display: flex; justify-content: space-around; margin: 20px 0;">
        <div style="background: #0f3460; padding: 20px; border-radius: 10px; text-align: center; width: 28%;">
            <h3 style="color: #3498db;">Accuracy</h3>
            <p style="font-size: 32px; margin: 0;"><b>{final_acc:.2f}%</b></p>
        </div>
        <div style="background: #0f3460; padding: 20px; border-radius: 10px; text-align: center; width: 28%;">
            <h3 style="color: #e74c3c;">Sensitivity</h3>
            <p style="font-size: 32px; margin: 0;"><b>{final_sens:.2f}%</b></p>
        </div>
        <div style="background: #0f3460; padding: 20px; border-radius: 10px; text-align: center; width: 28%;">
            <h3 style="color: #9b59b6;">Dice Score</h3>
            <p style="font-size: 32px; margin: 0;"><b>{final_dice:.2f}%</b></p>
        </div>
    </div>
    
    <h2 style="color: #f1c40f; margin-top: 30px;">💾 Saved Outputs</h2>
    <ul style="font-size: 14px;">
        <li>Model: <code>{Config.MODEL_DIR}/mobilenet_best.pth</code></li>
        <li>Training History: <code>{Config.OUTPUT_DIR}/training_history.png</code></li>
        <li>Confusion Matrix: <code>{Config.OUTPUT_DIR}/confusion_matrix.png</code></li>
        <li>Predictions: <code>{Config.OUTPUT_DIR}/predictions_visualization.png</code></li>
    </ul>
    
    <hr style="border-color: #00d4ff; margin-top: 30px;">
    <p style="text-align: center; color: #95a5a6;">✅ Training completed successfully on Kaggle Free Tier GPU</p>
    
</div>
"""

display(HTML(html_summary))

print("\n" + "="*70)
print("🎉 LIVER CANCER CLASSIFICATION COMPLETE!")
print("="*70)

In [ ]:
# ============================================================================
# CELL 21: SAVE MODEL FOR DEPLOYMENT
# ============================================================================

import json
import torch

# Save final model with metadata
model_info = {
    'architecture': 'MobileNet-V2',
    'num_classes': Config.NUM_CLASSES,
    'class_names': Config.CLASS_NAMES,
    'image_size': Config.IMAGE_SIZE,
    'preprocessing': {
        'hu_window': [Config.HU_MIN, Config.HU_MAX],
        'clahe_clip_limit': Config.CLAHE_CLIP_LIMIT,
        'normalization_mean': [0.485, 0.456, 0.406],
        'normalization_std': [0.229, 0.224, 0.225]
    },
    'metrics': {
        'test_accuracy': float(final_acc),
        'test_sensitivity': float(final_sens), 
        'test_dice': float(final_dice)
    },
    'training': {
        'epochs': Config.EPOCHS,
        'learning_rate': Config.LEARNING_RATE,
        'batch_size': Config.BATCH_SIZE
    }
}

# Save model weights
torch.save(model.state_dict(), f"{Config.MODEL_DIR}/liver_cancer_model.pth")
print(f"✅ Model weights saved to: {Config.MODEL_DIR}/liver_cancer_model.pth")

# Save complete model (architecture + weights) - for easy deployment
torch.save(model, f"{Config.MODEL_DIR}/liver_cancer_model_full.pth")
print(f"✅ Complete model saved to: {Config.MODEL_DIR}/liver_cancer_model_full.pth")

# Save metadata
with open(f"{Config.MODEL_DIR}/model_info.json", 'w') as f:
    json.dump(model_info, f, indent=2)
print(f"✅ Model metadata saved to: {Config.MODEL_DIR}/model_info.json")

# Save configuration as pickle for easy loading
import pickle
with open(f"{Config.MODEL_DIR}/config.pkl", 'wb') as f:
    pickle.dump(Config, f)
print(f"✅ Configuration saved to: {Config.MODEL_DIR}/config.pkl")

print(f"\n{'='*70}")
print("📦 MODEL DEPLOYMENT PACKAGE READY!")
print(f"{'='*70}")
print(f"\n📁 Saved files in '{Config.MODEL_DIR}/':")
print(f"   1. liver_cancer_model.pth      - Model weights (load with state_dict)")
print(f"   2. liver_cancer_model_full.pth - Complete model (load with torch.load)")
print(f"   3. model_info.json             - Model configuration & metrics")
print(f"   4. config.pkl                  - Config object (Python pickle)")
print(f"\n💡 Next steps:")
print(f"   • Download these files from Kaggle")
print(f"   • Use the inference script to make predictions")
print(f"   • Deploy to web app, API, or desktop application")